# Day-count conventions (reference)

**Deep dive:** `DayCount` maps a calendar span to a **year fraction** for accrual and discounting. This notebook compares every built-in convention on the same date range and sketches typical market usage.


## Concept

- A **day-count convention** answers: “How many years of time passed between `start` and `end`?” for interest accrual.
- In finstack, `year_fraction(start, end)` treats **`end` as exclusive** (consistent with many internal cash-flow primitives): the accrual period is \([start, end)\).
- **ACT/360** — money-market standard (USD/EUR deposits, many swaps). Denominator 360 inflates the year fraction versus ACT/365 for the same span.
- **30/360 (US Bond Basis)** — common on **corporate and municipal bonds**; months treated as 30 days.
- **ACT/365 Fixed** — often used for **sterling** cash instruments (e.g. UK gilts-style conventions in many systems).
- **ACT/ACT (ISDA)** — **US Treasury** and many **sovereign** bonds use actual-day accrual with an actual-day year split; in finstack use `DayCount.ACT_ACT` for ISDA-style actual/actual.
- **ACT/ACT (ISMA)** needs a **coupon frequency** via `DayCountContext`.
- **BUS/252** needs a **holiday calendar** via `DayCountContext` (Brazil and some equity derivatives contexts).
- **ACT/365L** and **30E/360** appear in specific regional/product conventions — compare numerically against your term sheet.


## API walkthrough

### Enumerated conventions and `year_fraction`

`DayCount` exposes one object per convention. `from_name("act_360")` parses snake-case labels.


In [1]:
from datetime import date

try:
    from finstack.core.dates import (
        DayCount,
        DayCountContext,
        Tenor,
    )
except ImportError as e:
    print("Import failed:", e)
else:
    all_dc = [
        DayCount.ACT_360,
        DayCount.ACT_365F,
        DayCount.ACT_365L,
        DayCount.THIRTY_360,
        DayCount.THIRTY_E_360,
        DayCount.ACT_ACT,
        DayCount.ACT_ACT_ISMA,
        DayCount.BUS_252,
    ]
    print("Built-in DayCount variants (str):")
    for dc in all_dc:
        print(" ", dc)
    parsed = DayCount.from_name("act_360")
    print("from_name('act_360') == ACT_360:", parsed == DayCount.ACT_360)


Built-in DayCount variants (str):
  act_360
  act_365f
  act_365l
  30_360
  30e_360
  act_act
  act_act_isma
  bus_252
from_name('act_360') == ACT_360: True


### Comparison table (same two dates)

The loop below is the canonical pattern: one `start`/`end`, every convention, formatted for quick side-by-side review. `ACT_ACT_ISMA` and `BUS_252` require `DayCountContext`.


In [2]:
from datetime import date

try:
    from finstack.core.dates import DayCount, DayCountContext, Tenor
except ImportError as e:
    print("Import failed:", e)
else:
    start, end = date(2024, 1, 15), date(2024, 7, 15)
    print(f"Interval [start, end) with end EXCLUSIVE: {start} -> {end}")
    print("convention year_fraction")
    print("-" * 40)
    ctx_isma = DayCountContext(calendar_id="usny", frequency=Tenor.parse("6M"))
    ctx_bus = DayCountContext(calendar_id="usny")
    rows = [
        (DayCount.ACT_360, None),
        (DayCount.ACT_365F, None),
        (DayCount.ACT_365L, None),
        (DayCount.THIRTY_360, None),
        (DayCount.THIRTY_E_360, None),
        (DayCount.ACT_ACT, None),
        (DayCount.ACT_ACT_ISMA, ctx_isma),
        (DayCount.BUS_252, ctx_bus),
    ]
    for dc, ctx in rows:
        yf = dc.year_fraction(start, end, ctx) if ctx is not None else dc.year_fraction(start, end)
        print(f"{dc!s:14s} {yf:.6f}")
    # User-requested snippet style
    print("---")
    conventions = [DayCount.ACT_360, DayCount.ACT_365F, DayCount.THIRTY_360]
    for dc in conventions:
        yf = dc.year_fraction(start, end)
        print(f"{dc}: {yf:.6f}")


Interval [start, end) with end EXCLUSIVE: 2024-01-15 -> 2024-07-15
convention year_fraction
----------------------------------------
act_360        0.505556
act_365f       0.498630
act_365l       0.497268
30_360         0.500000
30e_360        0.500000
act_act        0.497268
act_act_isma   0.500000
bus_252        0.496032
---
act_360: 0.505556
act_365f: 0.498630
30_360: 0.500000


### `calendar_days` helper

`DayCount.calendar_days` counts whole calendar days between dates (utility for checks and diagnostics).


In [3]:
from datetime import date

try:
    from finstack.core.dates import DayCount
except ImportError as e:
    print("Import failed:", e)
else:
    a, b = date(2024, 1, 1), date(2024, 1, 31)
    cd = DayCount.calendar_days(a, b)
    print(f"calendar_days({a}, {b}) = {cd}")


calendar_days(2024-01-01, 2024-01-31) = 30


## Practical example

Accrue a notional for **six months** under **ACT/360** vs **ACT/365F** (same exclusive end date). The ratio of year fractions matches the ratio of quoted accruals if the rate is identical.


In [4]:
from datetime import date

try:
    from finstack.core.dates import DayCount
except ImportError as e:
    print("Import failed:", e)
else:
    start, end = date(2024, 1, 15), date(2024, 7, 15)
    notional = 10_000_000.0
    rate = 0.05
    yf_360 = DayCount.ACT_360.year_fraction(start, end)
    yf_365f = DayCount.ACT_365F.year_fraction(start, end)
    accrual_360 = notional * rate * yf_360
    accrual_365f = notional * rate * yf_365f
    print(f"ACT/360  year_fraction={yf_360:.6f} accrual={accrual_360:,.2f}")
    print(f"ACT/365F year_fraction={yf_365f:.6f} accrual={accrual_365f:,.2f}")
    print(f"Ratio ACT360/ACT365F year fractions: {yf_360 / yf_365f:.6f}")


ACT/360  year_fraction=0.505556 accrual=252,777.78
ACT/365F year_fraction=0.498630 accrual=249,315.07
Ratio ACT360/ACT365F year fractions: 1.013889


## Takeaways

- Use **`year_fraction`** with the convention written in your term sheet; remember **`end` is exclusive** in finstack.
- **Money markets / USD-OIS-style short rates** → often **ACT/360**; **sterling fixed income** contexts frequently use **ACT/365F**; **US corporates** often **30/360**; **Treasuries / ISDA actual/actual** → **ACT_ACT**; **ISMA-style** fixed coupons → **ACT_ACT_ISMA** + frequency; **BUS/252** → calendar in `DayCountContext`.
- When migrating from another library, reconcile **exclusive vs inclusive end dates** before comparing year fractions.
